# Exploratory notebook template

Use this notebook in `scratch/notebooks/` to pull in artifacts from `tasks/.../output` and explore them.

A few ground rules for this repo:
- Read stable inputs from task outputs, or from downstream `input/` symlinks when you want to inspect exactly what a task consumes.
- Treat work here as exploratory. If a table, figure, or dataset becomes part of the workflow, move it into a proper task under `tasks/`.
- Keep paths relative to the repo structure rather than hardcoding machine-specific paths.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Find the repo root and load parameters

This assumes the notebook kernel starts somewhere inside the repository. The loop walks up until it finds `config/params.yaml`.


In [2]:
repo_root = Path.cwd().resolve()

while not (repo_root / "config" / "params.yaml").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not find repo root. Start Jupyter from somewhere inside this repo.")
    repo_root = repo_root.parent

scratch_dir = repo_root / "scratch" / "notebooks"
config_path = repo_root / "config" / "params.yaml"

print(f"repo_root: {repo_root}")
print(f"scratch_dir: {scratch_dir}")
print(f"config_path: {config_path}")


repo_root: /Users/connorbrennan/Library/CloudStorage/OneDrive-TheUniversityofChicago/Research/mmb_clean
scratch_dir: /Users/connorbrennan/Library/CloudStorage/OneDrive-TheUniversityofChicago/Research/mmb_clean/scratch/notebooks
config_path: /Users/connorbrennan/Library/CloudStorage/OneDrive-TheUniversityofChicago/Research/mmb_clean/config/params.yaml


In [3]:
with open(config_path, "r", encoding="utf-8") as handle:
    params = yaml.safe_load(handle)

params


{'project_name': 'mmb_clean',
 'default_language': 'python',
 'mmb': {'qforgive': 4,
  'horizons': [20, 40, 60],
  'drop_model_rule': 1,
  'pi_in_sacratio': 0,
  'time_limit': 60,
  'cloud_graph_extent': 20,
  'random_seed': 42,
  'duplicate_models_to_drop': ['US_ACELswt',
   'US_CMR10fa',
   'US_CMR14noFA',
   'US_MI07',
   'US_PM08',
   'US_VI16bgg',
   'US_VMDop',
   'US_YR13',
   'NK_GK09lin'],
  'excluded_models_to_drop': ['NK_CKL09',
   'NK_GM05',
   'NK_GM16',
   'NK_GS14',
   'NK_JO15_ht',
   'NK_CGG99',
   'US_IR15',
   'RBC_DTT11'],
  'sacrifice_ratio_models_to_drop': ['NK_CKL09',
   'NK_GM05',
   'NK_GS14',
   'NK_JO15_ht',
   'NK_CGG99',
   'US_IR15'],
  'output_response_labels': ['Output Gap']},
 'classifying_v2': {'gemini_model': 'gemini-3.5-flash',
  'cache_ttl_seconds': 172800,
  'temperature': 1.0,
  'max_output_tokens': 8192,
  'file_processing_poll_seconds': 2,
  'file_processing_timeout_seconds': 600,
  'request_pause_seconds': 1,
  'request_retry_attempts': 4,
  'r

## Point to the artifact you want to explore

Replace `dataset_relpath` with the task output you want. The example below points at a real artifact in this repo.

If you want to inspect the exact file a downstream task sees, you can instead point at something like `tasks/<task>/input/<file>`.


In [4]:
dataset_relpath = Path("tasks/data/build_mmb_analysis_dataset/output/MMB_reg_format.xlsx")
dataset_path = repo_root / dataset_relpath

if not dataset_path.exists():
    raise FileNotFoundError(dataset_path)

dataset_path


PosixPath('/Users/connorbrennan/Library/CloudStorage/OneDrive-TheUniversityofChicago/Research/mmb_clean/tasks/data/build_mmb_analysis_dataset/output/MMB_reg_format.xlsx')

## Load the data

Add another branch if you regularly work with another file type.


In [6]:
if dataset_path.suffix == ".parquet":
    df = pd.read_parquet(dataset_path)
elif dataset_path.suffix == ".feather":
    df = pd.read_feather(dataset_path)
elif dataset_path.suffix == ".dta":
    df = pd.read_stata(dataset_path)
elif dataset_path.suffix == ".pkl":
    df = pd.read_pickle(dataset_path)
elif dataset_path.suffix == ".csv" or dataset_path.name.endswith(".csv.gz"):
    df = pd.read_csv(dataset_path)
elif dataset_path.suffix == ".xlsx" or dataset_path.name.endswith(".xlsx.gz"):
    df = pd.read_excel(dataset_path)
else:
    raise ValueError(f"Add a reader for {dataset_path.name}")

print(df.shape)
df.head()


(222, 135)


,model,rule,rule_tr,rule_itr,rule_g,piq_value_min,y_value_min,irate_value_min,rrate_value_min,pi_value_min,piq_value_max,y_value_max,irate_value_max,rrate_value_max,pi_value_max,piq_timing_min,y_timing_min,irate_timing_min,rrate_timing_min,pi_timing_min,piq_timing_max,y_timing_max,irate_timing_max,rrate_timing_max,pi_timing_max,piq_cum20,y_cum20,irate_cum20,rrate_cum20,pi_cum20,piq_cum40,y_cum40,irate_cum40,rrate_cum40,pi_cum40,piq_cum60,y_cum60,irate_cum60,rrate_cum60,pi_cum60,sacratio20,sacratio40,sacratio60,flag_piq_wrongsign,flag_y_wrongsign,flag_pi_wrongsign,piq_chg_sign,y_chg_sign,irate_chg_sign,rrate_chg_sign,pi_chg_sign,irate_shock,rrate_shock,cb_authors,cb_authors_ext,estimated,calibrated,neq,open,firm_bs,bank,hh_demand,labor_frict,gov_spend,gov_debt,tax,fiscal,other_channel,learning,pr_ndx,wg_ndx,wg_ndx_mult,wg_ndx_prprice,wg_ndx_other,stky_wg,stky_pr,stky_pr_other,stky_pr_rotemberg,stky_pr_calvo,not_stky_pr,stky_pr_ndx,stky_wg_ndx,vint_early,vint_mid,vint_late,est_early,est_late,pub_date,est_start,est_end,piq_value_peak,y_value_peak,irate_value_peak,rrate_value_peak,pi_value_peak,piq_timing_peak,y_timing_peak,irate_timing_peak,rrate_timing_peak,pi_timing_peak,piq_timing_theorypeak,piq_value_theorypeak,y_timing_theorypeak,y_value_theorypeak,irate_timing_theorypeak,irate_value_theorypeak,rrate_timing_theorypeak,rrate_value_theorypeak,pi_timing_theorypeak,pi_value_theorypeak,piq_timing_agnpeak,piq_value_agnpeak,y_timing_agnpeak,y_value_agnpeak,irate_timing_agnpeak,irate_value_agnpeak,rrate_timing_agnpeak,rrate_value_agnpeak,pi_timing_agnpeak,pi_value_agnpeak,IScurve20,infl_per_rr20,Billsacrat20,IScurve40,infl_per_rr40,Billsacrat40,IScurve60,infl_per_rr60,Billsacrat60,stky_pr_nondx,stky_wg_nondx,stky_all,stky_pr_wg_ndx,ndx_all,ln_neq
0,G2_SIGMA08,Growth,0,0,1,-7.700607e-03,-3.509796e-02,-0.969323,-1.145733,-7.439740e-03,0.200913,0.223788,3.375478e-02,0.016204,0.181540,25.0,19.0,1,1,26.0,3,3,9.0,10.0,5,1.230085,1.066166,-2.244364,-3.363323,1.228472,1.168722,0.981979,-2.240582,-3.303605,1.166504,1.175892,0.969404,-2.236728,-3.304828,1.176640,5.713514,5.168803,5.243871,0,0,0,3,3,4,7,3,-0.969323,-1.145733,1.00,1.0,0.0,1.0,72.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,NaN,NaN,2008q3,.,.,0.200913,0.223788,-0.969323,-1.145733,0.181540,3,3,1,1,5,3,0.200913,3,0.223788,9.0,3.375478e-02,10.0,0.016204,5,0.181540,3,0.200913,3,0.223788,1,-0.969323,1,-1.145733,5,0.181540,-0.316998,-0.365735,1.153747,-0.297245,-0.353772,1.190170,-0.293330,-0.355810,1.213005,0.0,0.0,1.0,1.0,1.0,4.276666
1,G3_CW03,Growth,0,0,1,-4.648895e-03,-4.516538e-02,-0.993665,-1.039293,-4.541145e-03,0.093552,0.213891,4.296364e-02,0.024946,0.091423,29.0,18.0,1,1,31.0,7,5,12.0,21.0,9,1.137192,1.325714,-2.504020,-3.619038,1.117420,1.094357,1.085770,-2.347412,-3.412522,1.096365,1.094536,1.102529,-2.358702,-3.424931,1.094361,6.599834,6.735328,6.726483,0,0,0,4,4,4,5,4,-0.993665,-1.039293,1.00,1.0,1.0,0.0,9.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2002q3,1980q1,1998q4,0.093552,0.213891,-0.993665,-1.039293,0.091423,7,5,1,1,9,7,0.093552,5,0.213891,12.0,4.296364e-02,21.0,0.024946,9,0.091423,7,0.093552,5,0.213891,1,-0.993665,1,-1.039293,9,0.091423,-0.366317,-0.314225,0.857796,-0.318172,-0.320689,1.007909,-0.321913,-0.319579,0.992750,1.0,0.0,1.0,1.0,0.0,2.197225
2,G7_TAY93,Growth,0,0,1,3.730030e-04,1.336320e-04,-0.966616,-1.075820,3.967510e-04,0.167967,0.312177,6.497020e-02,0.025327,0.151010,99.0,99.0,1,1,99.0,4,3,12.0,15.0,6,1.531107,2.543254,-1.822828,-3.317302,1.498760,1.723014,2.658865,-1.450580,-3.123613,1.715223,1.785674,2.683124,-1.335161,-3.067959,1.782654,NaN,NaN,NaN,0,0,0,0,0,1,2,0,-0.966616,-1.075820,0.00,0.0,1.0,0.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1993q1,1971q1,1986q4,0.167967,0.312177,-0.966616,-1.075820,0.151010,4,3,1,1,6,4,0.167967,3,

In [7]:
df['test'] = df['estimated'] + df['calibrated']
print(f'Min is {df['test'].min()} and max is {df['test'].max()}')

Min is 1.0 and max is 1.0


## First-pass inspection

These are the basic checks I usually want immediately.


In [ ]:
df.info()


In [ ]:
pd.Series(df.columns, name="column").to_frame().head(50)


In [ ]:
missing_share = df.isna().mean().sort_values(ascending=False)
missing_share.head(25)


## Make a working copy and start exploring

Keep the raw loaded object untouched when possible.


In [ ]:
work = df.copy()

# Examples:
# work = work.loc[work["fyearq"].between(2000, 2020)].copy()
# work = work.loc[work["sic"].between(2000, 3999)].copy()
# work["leverage"] = (work["dlttq"].fillna(0) + work["dlcq"].fillna(0)) / work["atq"]
# work["log_atq"] = np.log(work["atq"])

work.head()


In [ ]:
summary_cols = [col for col in ["atq", "saleq", "capxy"] if col in work.columns]

if summary_cols:
    work[summary_cols].describe().T
else:
    print("Set summary_cols to variables that exist in this dataset.")


In [ ]:
plot_col = "atq"

if plot_col in work.columns:
    work[plot_col].dropna().hist(bins=50, figsize=(7, 4))
    plt.title(plot_col)
    plt.show()
else:
    print(f"Set plot_col to a column in this dataset. {plot_col!r} is not present.")


## When the notebook stops being exploratory

Once you know you need a stable figure, table, cleaned sample, or derived dataset, move that logic into a task under `tasks/` so it becomes reproducible and available downstream.
